In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Chybová mitigace pomocí tenzorových sítí (TEM): Qiskit funkce od Algorithmiq
*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (přes IBM Quantum Platform API) Plan. Jsou ve stavu preview release a mohou se změnit.


<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Přehled
Metoda chybové mitigace pomocí tenzorových sítí (TEM) od Algorithmiq je hybridní
kvantově-klasický algoritmus navržený pro provádění mitigace šumu výhradně ve
fázi klasického post-processingu. S TEM lze vypočítat střední hodnoty
pozorovatelných veličin a zmírnit nevyhnutelné chyby způsobené šumem na
kvantovém hardware s vyšší přesností a efektivitou nákladů,
což z něj dělá velmi atraktivní volbu pro kvantové výzkumníky i průmyslové
odborníky.

Metoda spočívá v sestavení tenzorové sítě reprezentující inverzi
globálního kanálu šumu ovlivňujícího stav kvantového procesoru a následném
aplikování tohoto zobrazení na informačně úplné výsledky měření získané
z zašuměného stavu, čímž se získají nestranné odhadce pozorovatelných veličin.

Jako výhodu TEM využívá informačně úplná měření pro přístup k rozsáhlé sadě
mitigovaných středních hodnot pozorovatelných veličin a dosahuje optimální
vzorkovací režie na kvantovém hardware, jak je popsáno ve Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), a Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Vzorkovací
režie označuje počet dodatečných měření potřebných k provedení efektivní
chybové mitigace, což je klíčový faktor pro proveditelnost kvantových výpočtů.
TEM má proto potenciál umožnit kvantovou výhodu ve složitých scénářích, jako jsou
aplikace v oblasti kvantového chaosu, fyziky mnoha těles, Hubbardovy dynamiky
a simulací chemie malých molekul.

Hlavní vlastnosti a výhody TEM lze shrnout takto:

1.  **Optimální vzorkovací režie**: TEM je optimální vzhledem k
[teoretickým hranicím](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
což znamená, že žádná metoda nemůže dosáhnout menší vzorkovací režie. Jinými
slovy, TEM vyžaduje minimální počet dodatečných měření pro provedení
chybové mitigace. To zároveň znamená, že TEM používá minimální kvantový runtime.
2.  **Nákladová efektivita**: Protože TEM zpracovává mitigaci šumu výhradně ve
fázi post-processingu, není potřeba přidávat do kvantového počítače extra Circuit,
což nejen snižuje náklady na výpočet, ale také snižuje riziko zavlečení
dodatečných chyb způsobených nedokonalostmi kvantových zařízení.
3.  **Odhad více pozorovatelných veličin**: Díky informačně úplným měřením TEM
efektivně odhaduje více pozorovatelných veličin ze stejných dat měření získaných
z kvantového počítače.
4.  **Mitigace chyb měření**: Qiskit Function TEM také zahrnuje
[proprietární metodu mitigace chyb měření](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
která dokáže výrazně snížit chyby čtení po krátkém kalibračním běhu.
5.  **Přesnost**: TEM výrazně zlepšuje přesnost a spolehlivost digitálních
kvantových simulací, čímž dělá kvantové algoritmy přesnějšími a spolehlivějšími.
## Popis
Funkce TEM ti umožňuje získat mitigované střední hodnoty pro
více pozorovatelných veličin na kvantovém Circuit s minimální vzorkovací režií.
Circuit je měřen pomocí informačně úplného měření s pozitivně semidefinitní
operátorovou hodnotou (IC-POVM) a získané výsledky měření jsou zpracovány
na klasickém počítači. Toto měření se používá k provedení metod tenzorových sítí
a sestavení mapy inverze šumu. Funkce aplikuje zobrazení, které plně invertuje
celý zašuměný Circuit pomocí tenzorových sítí reprezentujících zašuměné vrstvy.

![Schéma TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Mitigovaný odhad pozorovatelné O pomocí post-processingu výsledků měření zašuměného kvantového procesoru. U a N označují ideální kvantovou operaci a přidružené zobrazení šumu, které může být obecně nelokální (rozšířeno na šedé boxy). D označuje tenzor operátorů duálních k efektům v IC měření. Modul mitigace šumu M je tenzorová síť efektivně kontrahovaná od středu. První iterace kontrakce je znázorněna tečkovanou fialovou čarou, druhá přerušovanou čarou a třetí plnou čarou.")

Jakmile jsou Circuit odeslány do funkce, jsou transpilovaný a
optimalizovány tak, aby se minimalizoval počet vrstev s dvou-Qubitovými Gate
(nejhlučnějšími Gate na kvantových zařízeních). Šum ovlivňující vrstvy je
naučen prostřednictvím
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
pomocí řídkého Pauli-Lindbladova modelu šumu, jak je popsáno v E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model šumu je přesný popis šumu na zařízení schopný zachytit jemné rysy,
včetně přeslechů Qubitů. Šum na zařízeních však může kolísat a driftovat
a naučený šum nemusí být přesný v okamžiku provádění odhadu. To může vést
k nepřesným výsledkům.
## Začínáme
Proveď autentizaci pomocí svého [klíče IBM Quantum Platform API](http://quantum.cloud.ibm.com/) a vyber funkci TEM takto. (Tento úryvek předpokládá, že sis již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do lokálního prostředí.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load your function
tem = catalog.load(tem_function_name)

Ke kontrole stavu tvé Qiskit Function pracovní zátěže použij Qiskit Serverless API:

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use the Qiskit Serverless APIs to check your Qiskit Function workload's status:

In [3]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())

30db31ed-d252-4ca8-a5f0-5eb9b5075a4c
QUEUED


Výsledky lze vrátit takto:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.09417719217134088


<Admonition type="info">
  The expected value for the noiseless circuit for the given operator should be around `0.18409094298943401`.
</Admonition>

## Get support

Reach out to [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi)

Be sure to include the following information:
- Qiskit Function Job ID (`qiskit-ibm-catalog`), `job.job_id`
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
- Visit the [API reference](/docs/api/functions/algorithmiq-tem) for this Qiskit Function.

</Admonition>